In [4]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("amazon_delivery.csv", sep=';')
print("Ukuran data asli:", df.shape)
df.head()

Ukuran data asli: (43739, 16)


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,"4,9","22,745049","75,892471","22,765049","75,912471",19/03/2022,11.30.00,11.45.00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,"4,5","12,913041","77,683237","13,043041","77,813237",25/03/2022,19.45.00,19.50.00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,"4,4","12,914264","77,6784","12,924264","77,6884",19/03/2022,08.30.00,08.45.00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,"4,7","11,003669","76,976494","11,053669","77,026494",05/04/2022,18.00.00,18.10.00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,"4,6","12,972793","80,249982","13,012793","80,289982",26/03/2022,13.30.00,13.45.00,Cloudy,High,scooter,Metropolitian,150,Toys


In [5]:
print("Data kosong per kolom:")
print(df.isnull().sum())

print("\nTipe data:")
print(df.dtypes)

Data kosong per kolom:
Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather            91
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

Tipe data:
Order_ID             str
Agent_Age          int64
Agent_Rating         str
Store_Latitude       str
Store_Longitude      str
Drop_Latitude        str
Drop_Longitude       str
Order_Date           str
Order_Time           str
Pickup_Time          str
Weather              str
Traffic              str
Vehicle              str
Area                 str
Delivery_Time      int64
Category             str
dtype: object


In [11]:
import numpy as np
import pandas as pd

df = pd.read_csv('amazon_delivery.csv', sep=';') 

df.columns = df.columns.str.strip()

kolom_desimal = ['Agent_Rating', 'Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude']
for col in kolom_desimal:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

df['Late_delivery_risk'] = np.where(df['Delivery_Time'] > 120, 1, 0)

df['Order_Date_dt'] = pd.to_datetime(df['Order_Date'].astype(str).str.strip(), format='mixed', errors='coerce')
df['Is_Weekend'] = df['Order_Date_dt'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

order_time_clean = df['Order_Time'].astype(str).str.strip().str.replace('.', ':', regex=False)
pickup_time_clean = df['Pickup_Time'].astype(str).str.strip().str.replace('.', ':', regex=False)

df['Order_Time_dt'] = pd.to_datetime('2026-01-01 ' + order_time_clean, format='mixed', errors='coerce')
df['Pickup_Time_dt'] = pd.to_datetime('2026-01-01 ' + pickup_time_clean, format='mixed', errors='coerce')

df['Prep_Time_Minutes'] = (df['Pickup_Time_dt'] - df['Order_Time_dt']).dt.total_seconds() / 60

df['Distance_KM'] = (abs(df['Store_Latitude'] - df['Drop_Latitude'])
                     + abs(df['Store_Longitude'] - df['Drop_Longitude'])) * 111

kolom_dibuang = ['Order_ID', 'Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 
                 'Order_Date', 'Order_Time', 'Pickup_Time', 'Order_Time_dt', 'Pickup_Time_dt', 'Order_Date_dt', 'Delivery_Time']
df_cleaned = df.drop(columns=[col for col in kolom_dibuang if col in df.columns])

for col in df_cleaned.columns:
    if df_cleaned[col].isnull().sum() > 0:
        mode_series = df_cleaned[col].mode()
        if not mode_series.empty:
            df_cleaned[col] = df_cleaned[col].fillna(mode_series[0])

kolom_kategori = ['Weather', 'Traffic', 'Vehicle', 'Area', 'Category']
df_cleaned = pd.get_dummies(df_cleaned, columns=kolom_kategori, drop_first=True)

kolom_boolean = df_cleaned.select_dtypes(include='bool').columns
df_cleaned[kolom_boolean] = df_cleaned[kolom_boolean].astype(int)

print("Ukuran data setelah pembersihan & encoding:", df_cleaned.shape)
df_cleaned.head()

Ukuran data setelah pembersihan & encoding: (43739, 36)


,Agent_Age,Agent_Rating,Late_delivery_risk,Is_Weekend,Prep_Time_Minutes,Distance_KM,Weather_Fog,Weather_Sandstorms,Weather_Stormy,Weather_Sunny,...,Category_Home,Category_Jewelry,Category_Kitchen,Category_Outdoors,Category_Pet Supplies,Category_Shoes,Category_Skincare,Category_Snacks,Category_Sports,Category_Toys
0,37,4.9,0,1,15.0,4.44,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,34,4.5,1,0,5.0,28.86,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,23,4.4,1,1,15.0,2.22,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
3,38,4.7,0,0,10.0,11.10,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,32,4.6,1,1,15.0,8.88,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
